# Machine Learning-Based Network Intrusion Detection Using CIC-IDS2017

## 1. Project Overview

This project builds an end-to-end supervised Machine Learning pipeline for network intrusion detection using the **CIC-IDS2017** dataset. Network traffic can be represented as a collection of numerical flow-level features (packet counts, byte counts, timing statistics, flag counts, etc.), which makes it possible to frame intrusion detection as a **multi-class classification problem**: given a traffic flow's features, predict whether it is benign or one of several attack types.

The workflow proceeds from raw traffic data through exploratory analysis, leakage-aware preprocessing, feature selection, baseline modeling, hyperparameter tuning, multi-model comparison, and a detailed final evaluation. The result is a trained classifier together with a full record of the decisions, metrics, and artifacts produced along the way.

This is **not** presented as a perfect, production-ready, or universally best system — see the Limitations section for an honest account of what the results do and do not show.

## 2. Project Objective

The objective is to build a Machine Learning classification pipeline that distinguishes four encoded network-traffic classes in CIC-IDS2017, with emphasis on:

- Understanding the data before modeling (EDA)
- Data quality handling (missing values, infinite values, duplicates, constant features)
- Leakage-aware preprocessing (fitting all transformations on training data only)
- Dimensionality reduction and feature selection
- Class-imbalance-aware model configuration
- Systematic hyperparameter tuning
- Comparing multiple model families rather than committing to one algorithm upfront
- Detailed, class-wise evaluation rather than relying on aggregate accuracy alone
- Interpretability through feature importance

## 3. Problem Statement

Network intrusion detection on CIC-IDS2017 presents several practical challenges:

- **High-dimensional traffic data** — the raw dataset contains 78 numerical flow features.
- **Redundant features** — many flow statistics (e.g. subflow byte/packet counts) are near-duplicates of other features.
- **Outliers** — flow-level statistics such as duration and inter-arrival times are heavily skewed and contain extreme values.
- **Severe class imbalance** — benign traffic vastly outnumbers attack traffic, and one attack class in particular is extremely rare.
- **Minority-class detection** — a model can score well on accuracy while still failing to recognize rare attack types.
- **Model selection** — no single algorithm is obviously best in advance; several need to be compared on the same data.
- **Reliable evaluation** — results must be interpreted with attention to *how* the held-out data was used at each stage, not just the final metric.

## 4. Dataset Overview

| Item | Value |
|---|---|
| Dataset | CIC-IDS2017 |
| Raw file | `CIC_IDS2017.csv` |
| Raw shape | 1,233,163 rows × 79 columns |
| Original target column | `Label` (raw dataset) |
| Encoded target column | `Target_Encoded` |
| Number of classes | 4 |
| Encoded classes | 0, 1, 2, 3 |

The raw `Label` column contains four traffic categories: `BENIGN`, `Bot`, `PortScan`, and `DDoS`. During preprocessing these are label-encoded (`.astype("category").cat.codes`) into `Target_Encoded`, producing the following mapping (taken directly from Notebook 02's output):

| Encoded Class | Original Label |
|---|---|
| 0 | BENIGN |
| 1 | Bot |
| 2 | DDoS |
| 3 | PortScan |

Class **1 (Bot)** is the rarest category by a wide margin and is the main source of classification difficulty throughout the project.

## 5. Complete Project Architecture

```mermaid
flowchart TD
    A[CIC-IDS2017 Raw Dataset<br/>1,233,163 rows x 79 columns] --> B[Notebook 01 - EDA]
    B --> C[Notebook 02 - Preprocessing]
    C --> D[Notebook 03 - Feature Selection]
    D --> E[37 Preprocessed Features]
    E --> F[Variance Threshold]
    F --> G[Pearson Correlation Check]
    G --> H[Random Forest Feature Importance]
    H --> I[12 Final Selected Features]
    I --> J[Notebook 04 - Baseline Model Building]
    J --> K[Notebook 05 - Hyperparameter Tuning]
    K --> L[Notebook 06 - Model Comparison]
    L --> M[Decision Tree Selected]
    M --> N[Notebook 07 - Final Model Evaluation]
    N --> O[Performance Metrics]
    O --> P[Classification Report]
    P --> Q[Confusion Matrix]
    Q --> R[Error Analysis]
    R --> S[Feature Importance]
    S --> T[Final Model Saved]
    T --> U[Evaluation Results Saved]
    U --> V[Final Notebook Validation: True]
```

## 6. Project Notebook Structure

| Notebook | Purpose | Major Techniques | Main Output | Contribution |
|---|---|---|---|---|
| 01_EDA | Understand raw dataset structure and quality | Shape/dtype inspection, class distribution, missing/duplicate detection, skewness, IQR outlier scan, correlation heatmap | EDA findings; identified 114,482 duplicates, 10 zero-variance features, 71 highly correlated pairs (≥0.90) | Informs preprocessing and modeling decisions |
| 02_Preprocessing | Clean, encode, split, cap outliers, filter correlated features, standardize | Dedup, infinite→NaN, median imputation, label encoding, 80:20 stratified split, IQR capping (train-derived bounds), Pearson filtering (0.90), StandardScaler | `X_train.csv`, `X_test.csv`, `y_train.csv`, `y_test.csv`, `standard_scaler.pkl` (37 features) | Produces a leakage-aware, standardized dataset |
| 03_Feature_Selection | Reduce 37 features to the most informative subset | Variance threshold, Pearson correlation recheck, Random Forest importance (median threshold) | `selected_features.csv`; `X_train_selected`/`X_test_selected` (12 features) | Reduces dimensionality by 67.57% while retaining signal |
| 04_Model_Building | Establish a baseline classifier | Random Forest (`n_estimators=100, class_weight="balanced_subsample"`) | `random_forest_baseline.pkl`, `model_building_baseline_results.csv` | Baseline Macro F1 = 0.946513 |
| 05_Hyperparameter_Tuning | Systematically search Random Forest configurations | `RandomizedSearchCV`, 6 candidates, Stratified 3-fold CV, scoring = Macro F1 | `random_forest_tuned.pkl`, `random_forest_best_parameters.csv` | Best CV Macro F1 = 0.950453; test Macro F1 = 0.946215 |
| 06_Model_Comparison | Compare four distinct algorithms on the same data | Logistic Regression, Decision Tree, Random Forest, Gradient Boosting | Ranked comparison table | Decision Tree selected (highest test Macro F1 = 0.947471) |
| 07_Final_Model_Evaluation | Thorough evaluation of the selected model | Classification report, confusion matrix, error analysis, feature importance, artifact saving, final validation | `final_decision_tree.pkl`, four result CSVs | Final documented performance and interpretability |

The workflow proceeds strictly as **01 → 02 → 03 → 04 → 05 → 06 → 07**, with each stage consuming the artifacts produced by the previous one.

---

## 7. Notebook 01 — Exploratory Data Analysis

EDA was performed on the untouched raw dataset (1,233,163 rows × 79 columns) before any cleaning, to understand what preprocessing would actually be needed.

**Key findings:**

| Check | Result |
|---|---|
| Rows / Columns | 1,233,163 / 79 |
| Target column (raw) | `Label`, 4 unique classes |
| Raw class counts | BENIGN 944,240 · PortScan 158,930 · DDoS 128,027 · Bot 1,966 |
| Raw class percentages | BENIGN 76.57% · PortScan 12.89% · DDoS 10.38% · Bot 0.16% |
| Missing values | 111 (all in `Flow Bytes/s`) |
| Duplicate rows | 114,482 (9.28%) |
| Numerical features | 78 |
| Categorical features | 1 (the target) |
| Zero-variance features | 10 |
| Highly correlated pairs (\|r\| ≥ 0.90) | 71 |

EDA is important before modeling because it surfaces data-quality issues (missing/duplicate/constant values), reveals the severity of class imbalance, and identifies redundancy among features — all of which directly shape the preprocessing and feature-selection strategy used in later notebooks. Notebook 01's contribution is this initial map of the dataset's structure and problems, not a finished dataset.

## 8. Notebook 02 — Data Preprocessing

### 8.1 Data cleaning before splitting

Before the train/test split, the following cleaning was applied to the full dataset:

- Infinite values (1,817 of them) were replaced with `NaN`.
- Rows with a missing target label were dropped.
- Remaining missing numerical values were filled with the **median** of each feature.
- 114,482 duplicate rows were removed (1,233,163 → 1,118,681 rows).
- 10 constant (zero/near-zero variance) features were dropped (79 → 68 usable feature columns, after also removing the raw string label from the numeric feature set).
- The target was label-encoded into `Target_Encoded` using the mapping in Section 4.

### 8.2 Train/Test Split

An **80:20 stratified train-test split** was used, on the cleaned 1,118,681-row dataset with 68 features.

| Object | Shape |
|---|---|
| `X_train` | (894,944, 68) |
| `X_test` | (223,737, 68) |
| `y_train` | (894,944,) |
| `y_test` | (223,737,) |

An 80:20 split provides a large enough training set for a high-dimensional problem while reserving a substantial, statistically meaningful held-out set for evaluation. **Stratification** on the target ensures the (heavily imbalanced) class proportions are preserved in both subsets — without it, a random split could easily leave the test set with very few or zero examples of the rarest class.

### 8.3 Class Distribution After Splitting

**Training set:**

| Class | Count | Percentage |
|---|---|---|
| 0 (BENIGN) | 718,314 | 80.26% |
| 1 (Bot) | 1,562 | 0.17% |
| 2 (DDoS) | 102,413 | 11.44% |
| 3 (PortScan) | 72,655 | 8.12% |

**Testing set:**

| Class | Count | Percentage |
|---|---|---|
| 0 (BENIGN) | 179,579 | 80.26% |
| 1 (Bot) | 391 | 0.17% |
| 2 (DDoS) | 25,603 | 11.44% |
| 3 (PortScan) | 18,164 | 8.12% |

The nearly identical percentages across train and test confirm the stratified split worked as intended. Class 1 (Bot) is extremely underrepresented — only 1,562 training examples out of 894,944 — which is the central challenge for this project's modeling and evaluation strategy.

### 8.4 IQR Outlier Handling

Outliers were handled using the **Interquartile Range (IQR)** method, with an important methodological rule: **IQR bounds (Q1, Q3, and the resulting lower/upper bounds) were calculated using training data only.** The same training-derived bounds were then applied to both the training and testing feature matrices.

Rather than deleting rows, values outside the bounds were **capped (clipped)** to the boundary values. After capping, zero values remained outside the bounds on both the training and testing sets.

Capping (rather than deletion) was chosen because:

- It preserves the size of the dataset, including rare examples of minority-class traffic.
- Deleting rows could disproportionately remove already-scarce Class 1 (Bot) samples.
- It reduces the influence of extreme values without discarding whatever legitimate signal they contain.
- Fitting the bounds only on training data prevents information about the test distribution from leaking into the capping thresholds.

### 8.5 Correlation Filtering

A Pearson correlation threshold of **0.90** was applied to the 68-feature training matrix (computed on training data only, to avoid leakage).

| Metric | Value |
|---|---|
| Correlation threshold | 0.90 |
| Highly correlated pairs identified | 77 |
| Features removed | 31 |
| Features remaining | 37 |

Highly correlated features carry largely redundant information; removing them reduces dimensionality, computational cost, and multicollinearity-related instability, while improving interpretability — without meaningfully sacrificing predictive signal, since one feature from each correlated pair still remains.

### 8.6 Standardization

Features were standardized using **`StandardScaler`**, fit **only on the training data**. The fitted scaler was then used to transform both the training and testing sets.

| Check | Value |
|---|---|
| Maximum absolute training feature mean | ≈ 1.489 × 10⁻¹⁶ |
| Minimum training feature standard deviation | 0.0 |
| Maximum training feature standard deviation | ≈ 1.0000006 |
| Average absolute testing feature mean | ≈ 0.001129 |
| Average testing feature standard deviation | ≈ 0.621737 |
| Missing values after preprocessing | 0 |
| Infinite values after preprocessing | 0 |

Fitting the scaler on the combined training + testing data (or on testing data alone) would leak information about the test distribution into the preprocessing pipeline, producing an overly optimistic sense of how the model would perform on genuinely unseen data. Note that the testing set's mean/std are *not* exactly 0/1 — this is expected and correct, since the scaler's parameters come only from training data.

### 8.7 Preprocessing Output Files

| File | Purpose |
|---|---|
| `../data/processed/X_train.csv` | Standardized training features (894,944 × 37) |
| `../data/processed/X_test.csv` | Standardized testing features (223,737 × 37) |
| `../data/processed/y_train.csv` | Training target labels |
| `../data/processed/y_test.csv` | Testing target labels |
| `../data/processed/standard_scaler.pkl` | Fitted `StandardScaler` object (training-derived) |

---

## 9. Notebook 03 — Feature Selection

Starting from the 37 preprocessed features, a three-stage pipeline reduced the feature set to 12:

```
37 features → Variance Threshold → 23 → Pearson Correlation (recheck) → 23 → Random Forest Importance → 12
```

### 9.1 Variance Threshold

- Threshold: **0.0** (zero-variance features)
- Result: **37 → 23** features (14 zero-variance features removed)

A zero-variance feature has the same value across all observations and therefore carries no information that could help distinguish between classes — it can be safely dropped.

### 9.2 Pearson Correlation (Recheck)

- Threshold: |correlation| ≥ 0.90
- Result: **0 additional features removed**; 23 features remained

This is consistent with, not contradictory to, Notebook 02: correlation filtering had already been applied during preprocessing (68 → 37 features), so re-checking the smaller 23-feature variance-selected set found no remaining pairs above the 0.90 threshold.

### 9.3 Random Forest Feature Selection

```python
RandomForestClassifier(
    n_estimators=100,
    max_depth=20,
    class_weight="balanced_subsample",
    n_jobs=-1,
    random_state=42
)
```

- Selection method: features with importance ≥ the **median** importance value
- Median importance threshold: **0.034235**

**Selected features (12):**

1. Destination Port
2. Flow Duration
3. Total Fwd Packets
4. Total Length of Fwd Packets
5. Total Length of Bwd Packets
6. Fwd Packet Length Mean
7. Bwd Packet Length Min
8. Bwd Packets/s
9. PSH Flag Count
10. Init_Win_bytes_forward
11. Init_Win_bytes_backward
12. min_seg_size_forward

| Metric | Value |
|---|---|
| Final reduction | 37 → 12 |
| Reduction percentage | 67.57% |
| Retention percentage | 32.43% |

Feature selection at this scale improves computational efficiency, reduces redundancy, simplifies the resulting model, and focuses the classifier on the network-traffic characteristics that a Random Forest found most informative for distinguishing the four classes.

### 9.4 Feature Selection Outputs

`selected_features.csv` records the final 12 feature names along with their Random Forest importance scores, serving as the definitive feature list used by every subsequent notebook.

---

## 10. Notebook 04 — Baseline Model Building

**Baseline algorithm:**

```python
RandomForestClassifier(
    n_estimators=100,
    class_weight="balanced_subsample",
    n_jobs=-1,
    random_state=42
)
```

Trained on the 12 selected features. **Training time: 35.36 seconds.**

### 10.1 Baseline Overall Results

| Metric | Score | Percentage |
|---|---|---|
| Accuracy | 0.999057 | 99.9057% |
| Weighted Precision | 0.999252 | 99.9252% |
| Weighted Recall | 0.999057 | 99.9057% |
| Weighted F1 | 0.999122 | 99.9122% |
| Macro F1 | 0.946513 | 94.6513% |

Accuracy, precision, recall, and F1 are standard classification metrics; the "weighted" versions average per-class scores weighted by class support, so they are dominated by the huge majority class (Class 0). **Macro F1** instead averages F1 equally across all four classes regardless of size, so it is far more sensitive to how well the rare Class 1 (Bot) is detected — which is why it is the metric emphasized throughout this project.

### 10.2 Baseline Classification Results

| Class | Precision | Recall | F1 | Support |
|---|---|---|---|---|
| 0 | 1.00 | 1.00 | 1.00 | 179,579 |
| 1 | 0.68 | 0.93 | 0.79 | 391 |
| 2 | 1.00 | 1.00 | 1.00 | 25,603 |
| 3 | 1.00 | 1.00 | 1.00 | 18,164 |

Class 1 is already visibly the weak point: precision of only 0.68 means many predicted "Class 1" flows are actually something else, even though recall of 0.93 shows most true Class 1 flows are being found.

### 10.3 Baseline Confusion Matrix

| Actual \\ Predicted | Class 0 | Class 1 | Class 2 | Class 3 |
|---|---|---|---|---|
| **Class 0** | 179,403 | 167 | 8 | 1 |
| **Class 1** | 29 | 362 | 0 | 0 |
| **Class 2** | 4 | 0 | 25,599 | 0 |
| **Class 3** | 2 | 0 | 0 | 18,162 |

The dominant error pattern is confusion between Class 0 and Class 1 in both directions (167 Class-0-as-1, 29 Class-1-as-0), while Classes 2 and 3 are classified almost perfectly.

### 10.4 Baseline Outputs

- Model: `../models/random_forest_baseline.pkl`
- Results: `model_building_baseline_results.csv`

---

## 11. Notebook 05 — Hyperparameter Tuning

**Method:** `RandomizedSearchCV` with **6** randomly sampled hyperparameter configurations, using **Stratified 3-fold cross-validation**, scored on **Macro F1**. The held-out test set was **not** used during the search itself — only training data, via cross-validation.

**Search space:**

| Hyperparameter | Values |
|---|---|
| `n_estimators` | [100, 150, 200] |
| `max_depth` | [10, 20, 30, None] |
| `min_samples_split` | [2, 5, 10] |
| `min_samples_leaf` | [1, 2, 4] |
| `max_features` | ["sqrt", "log2"] |
| `class_weight` | ["balanced", "balanced_subsample"] |

- `n_estimators` — number of trees in the forest.
- `max_depth` — maximum depth each tree may grow to (controls overfitting).
- `min_samples_split` / `min_samples_leaf` — minimum samples required to split a node / to be a leaf (also control overfitting).
- `max_features` — number of features considered at each split.
- `class_weight` — how much extra weight minority classes receive during training.

`RandomizedSearchCV` was used instead of an exhaustive grid search because it samples a representative subset of the search space at a fraction of the computational cost. Stratified cross-validation keeps class proportions consistent across folds, which matters given the severe imbalance. Macro F1 was chosen as the scoring metric specifically so tuning would optimize for balanced class-wise performance rather than being dominated by the majority class.

**Result of the search (from `random_search.best_score_` / `random_search.best_params_`):**

| Item | Value |
|---|---|
| Candidates evaluated | 6 (× 3 folds = 18 fits) |
| Total tuning time | 715.44 seconds (≈ 11.92 minutes) |
| Best CV Macro F1 | **0.950453** |
| Best `n_estimators` | 200 |
| Best `min_samples_split` | 5 |
| Best `min_samples_leaf` | 1 |
| Best `max_features` | sqrt |
| Best `max_depth` | 20 |
| Best `class_weight` | balanced_subsample |

**Tuned model evaluated on the held-out test set:**

| Metric | Tuned RF | Baseline RF (for reference) |
|---|---|---|
| Accuracy | 0.999017 | 0.999057 |
| Weighted Precision | 0.999279 | 0.999252 |
| Weighted Recall | 0.999017 | 0.999057 |
| Weighted F1 | 0.999100 | 0.999122 |
| Macro F1 | **0.946215** | 0.946513 |

An honest observation from this notebook: although the tuned configuration achieved the best **cross-validated** Macro F1 (0.950453), its performance on the held-out test set (Macro F1 = 0.946215) was marginally **lower** than the untuned baseline (0.946513). This is a normal outcome of hyperparameter search — the best CV score does not always transfer exactly to the test set — and it is one reason the project moved on to compare multiple model families in Notebook 06 rather than committing to this tuned Random Forest.

- Tuned model saved to: `../models/random_forest_tuned.pkl`
- Best parameters saved to: `../data/processed/random_forest_best_parameters.csv`

---

## 12. Notebook 06 — Model Comparison

Four models were trained on the same 12-feature training set and evaluated on the same held-out test set:

```python
# Logistic Regression
LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42, n_jobs=-1)

# Decision Tree
DecisionTreeClassifier(class_weight="balanced", random_state=42)

# Random Forest
RandomForestClassifier(n_estimators=100, class_weight="balanced_subsample", n_jobs=-1, random_state=42)

# Gradient Boosting
GradientBoostingClassifier(n_estimators=100, random_state=42)
```

### 12.1 Model Comparison Results

| Model | Accuracy | Weighted Precision | Weighted Recall | Weighted F1 | Macro F1 | Training Time (s) | Prediction Time (s) |
|---|---|---|---|---|---|---|---|
| Decision Tree | 0.999169 | 0.999175 | 0.999169 | 0.999172 | **0.947471** | 7.16 | 0.036 |
| Gradient Boosting | 0.999280 | 0.999266 | 0.999280 | 0.999222 | 0.947336 | 558.28 | 1.46 |
| Random Forest | 0.999057 | 0.999252 | 0.999057 | 0.999122 | 0.946513 | 34.84 | 0.31 |
| Logistic Regression | 0.959600 | 0.994467 | 0.959600 | 0.975757 | 0.760096 | 16.57 | 0.03 |

- **Decision Tree** achieved the highest Macro F1 (0.947471) with by far the lowest training time (7.16 s) among the tree-based models.
- **Gradient Boosting** achieved marginally higher raw accuracy (0.999280) and a very close Macro F1 (0.947336), but took nearly 80× longer to train (558.28 s vs 7.16 s).
- **Random Forest** (baseline configuration) performed slightly below Decision Tree on Macro F1.
- **Logistic Regression**, as a linear model, was clearly outmatched (Macro F1 = 0.760096) on this non-linearly-separable, imbalanced problem.

### 12.2 Model Selection

The **Decision Tree** was selected as the final model **within this project**, based on:

- Highest Macro F1-score (0.947471) among the four candidates
- Class-imbalance-aware configuration (`class_weight="balanced"`)
- Substantially lower computational cost than Gradient Boosting, its closest competitor on Macro F1

This is not a claim that Decision Tree is universally the best algorithm for network intrusion detection — it is the model that performed best on this dataset, this feature set, and this evaluation criterion.

### 12.3 Important Methodological Caveat

Notebook 06 trained each of the four models on the training set and then evaluated and **ranked** them using the same held-out test set that was also used in Notebooks 04 and 05. This means the test set was used to make a model-selection decision, not purely to produce a final, one-time estimate of generalization performance.

**In plain terms:** the held-out evaluation dataset was used for model comparison and final performance analysis, not as a completely untouched, unbiased test set by the time Notebook 07 begins. This is a normal and common practice in applied ML projects, but it is worth stating plainly rather than implying the final numbers represent performance on data the model-selection process never touched.

---

## 13. Notebook 07 — Final Model Evaluation

**Final model:**

```python
DecisionTreeClassifier(class_weight="balanced", random_state=42)
```

| Item | Value |
|---|---|
| Final features | 12 |
| Training samples | 894,944 |
| Evaluation samples | 223,737 |

**Final selected features:** Destination Port, Flow Duration, Total Fwd Packets, Total Length of Fwd Packets, Total Length of Bwd Packets, Fwd Packet Length Mean, Bwd Packet Length Min, Bwd Packets/s, PSH Flag Count, Init_Win_bytes_forward, Init_Win_bytes_backward, min_seg_size_forward.

### 13.1 Final Model Pipeline

```mermaid
flowchart TD
    A[12 Selected Features] --> B[Decision Tree Classifier]
    B --> C[Training - 894,944 Samples]
    C --> D[Trained Final Model]
    D --> E[Evaluation Dataset - 223,737 Samples]
    E --> F[Predictions]
    F --> G[Performance Metrics]
    G --> H[Classification Report]
    H --> I[Confusion Matrix]
    I --> J[Error Analysis]
    J --> K[Feature Importance]
    K --> L[Saved Model: final_decision_tree.pkl]
    L --> M[Saved Evaluation Results]
```

### 13.2 Final Model Training

- Training time: **6.36 seconds**
- Saved model: `../models/final_decision_tree.pkl`

### 13.3 Final Model Prediction

- Prediction time: **0.07 seconds**
- Prediction shape: (223,737,)
- Predicted classes: [0, 1, 2, 3]
- Prediction count verification: True (223,737 predictions for 223,737 samples)

### 13.4 Final Overall Metrics

| Metric | Score | Percentage |
|---|---|---|
| Accuracy | 0.999169 | 99.9169% |
| Weighted Precision | 0.999175 | 99.9175% |
| Weighted Recall | 0.999169 | 99.9169% |
| Weighted F1 | 0.999172 | 99.9172% |
| Macro F1 | 0.947471 | 94.7471% |

- **Accuracy** — proportion of all predictions that were correct.
- **Precision** — of the flows predicted as a given class, what fraction actually belong to it.
- **Recall** — of the flows that truly belong to a given class, what fraction were found.
- **F1-score** — harmonic mean of precision and recall for a class.
- **Weighted F1** — F1 averaged across classes, weighted by class size (dominated by Class 0).
- **Macro F1** — F1 averaged equally across all four classes, regardless of size.

Macro F1 is especially important here because Class 1 (Bot) makes up only 0.17% of the data; a metric that isn't dominated by class size is required to see how the model actually performs on it. The model is **not** claimed to be perfect — Class 1's own numbers, below, show exactly where it falls short.

---

## 14. Final Classification Report

| Class | Precision | Recall | F1-score | Support |
|---|---|---|---|---|
| Class 0 | 0.999516 | 0.999449 | 0.999482 | 179,579 |
| Class 1 | 0.783920 | 0.797954 | 0.790875 | 391 |
| Class 2 | 0.999688 | 0.999805 | 0.999746 | 25,603 |
| Class 3 | 0.999725 | 0.999835 | 0.999780 | 18,164 |

**Overall:**

| Metric | Value |
|---|---|
| Accuracy | 0.999169 |
| Macro avg precision | 0.945712 |
| Macro avg recall | 0.949261 |
| Macro avg F1 | 0.947471 |
| Weighted avg precision | 0.999175 |
| Weighted avg recall | 0.999169 |
| Weighted avg F1 | 0.999172 |

Classes 0, 2, and 3 are all classified with F1 scores above 0.999. **Class 1 (Bot) is clearly the most difficult class**, with an F1 of only 0.790875 — roughly 21 points lower than any other class. This gap, not the headline accuracy figure, is the most important number in this evaluation: it shows that near-perfect overall accuracy is compatible with meaningfully weaker detection of the rarest attack type.

## 15. Final Confusion Matrix

| Actual \\ Predicted | Class 0 | Class 1 | Class 2 | Class 3 |
|---|---|---|---|---|
| **Class 0** | 179,480 | 86 | 8 | 5 |
| **Class 1** | 79 | 312 | 0 | 0 |
| **Class 2** | 5 | 0 | 25,598 | 0 |
| **Class 3** | 3 | 0 | 0 | 18,161 |

- Class 0: 179,480 of 179,579 correctly classified.
- Class 1: 312 of 391 correctly classified; 79 true Bot flows were misclassified as BENIGN.
- Class 2 and Class 3: essentially perfect, with only 5 and 3 misclassifications respectively.
- The dominant confusion, by far, is between Class 0 and Class 1 in both directions (86 Class-0-as-1, 79 Class-1-as-0) — consistent with Bot traffic sometimes resembling benign traffic at the flow-feature level used here.

## 16. Error Analysis

| Quantity | Value |
|---|---|
| Evaluation samples | 223,737 |
| Correct predictions | 223,551 |
| Incorrect predictions | 186 |
| Correct rate | 0.999169 |
| Error rate | 0.000831 (0.0831%) |

**Class-wise error breakdown:**

| Class | Actual | Correct | Incorrect | Error Rate |
|---|---|---|---|---|
| 0 | 179,579 | 179,480 | 99 | 0.000551 |
| 1 | 391 | 312 | 79 | **0.202046** |
| 2 | 25,603 | 25,598 | 5 | 0.000195 |
| 3 | 18,164 | 18,161 | 3 | 0.000165 |

The overall error count is very small (186 out of 223,737), but it is not evenly distributed: **Class 1 alone accounts for 79 of the 186 total errors (≈42%) despite making up only 0.17% of the evaluation set**, giving it an error rate of roughly 20.2% — over 300× higher than any other class. The majority classes have error rates below 0.06%. This is exactly why overall accuracy needs to be interpreted alongside Macro F1 and class-wise metrics rather than on its own.

## 17. Final Feature Importance

| Feature | Importance |
|---|---|
| Fwd Packet Length Mean | 0.394534 |
| Destination Port | 0.251689 |
| Total Length of Bwd Packets | 0.122199 |
| Total Length of Fwd Packets | 0.072831 |
| Init_Win_bytes_backward | 0.049323 |
| PSH Flag Count | 0.048345 |
| Init_Win_bytes_forward | 0.034304 |
| Bwd Packets/s | 0.022367 |
| Flow Duration | 0.003832 |
| Total Fwd Packets | 0.000367 |
| min_seg_size_forward | 0.000138 |
| Bwd Packet Length Min | 0.000071 |

- Total importance: 1.000000 · Mean importance: 0.083333
- Most important: **Fwd Packet Length Mean** (0.394534)
- Second: **Destination Port** (0.251689)
- Third: **Total Length of Bwd Packets** (0.122199)

Feature importance here reflects how much the Decision Tree relied on each feature when choosing splits during training — it is a measure of **model-specific contribution**, not a measure of real-world causality. It would be incorrect to claim, for instance, that a particular destination port "causes" an intrusion; it only means that this feature was useful for this model in distinguishing the four classes in this dataset.

## 18. Data Leakage Prevention

The pipeline deliberately separates "learn from training data" steps from "apply to both sets" steps at every stage:

- 80:20 **stratified** train-test split performed once, up front.
- IQR bounds calculated **from training data only**, then the same bounds applied to both training and testing data.
- Correlation analysis (both the 0.90-threshold filtering in preprocessing and the recheck in feature selection) performed using **training data only**.
- `StandardScaler` fit **only on training data**; the same fitted scaler transforms both sets.
- Hyperparameter tuning used training data with **stratified cross-validation**; the held-out evaluation set was not used to select hyperparameters.
- **Exception, stated plainly:** Notebook 06 *did* use the held-out evaluation data to compare and select among the four candidate models.

```mermaid
flowchart TD
    A[Raw Dataset] --> B[Stratified Train/Test Split]
    B --> C[Training Data]
    B --> D[Held-Out Evaluation Data]
    C --> E[Calculate IQR Bounds]
    D --> F[Apply Training-Derived IQR Bounds]
    C --> G[Correlation Analysis]
    D --> H[Apply Selected Features]
    C --> I[Fit StandardScaler]
    D --> J[Transform Using Same Scaler]
    C --> K[Feature Selection]
    K --> L[Model Training / Cross-Validation]
    F --> M[Evaluation]
    H --> M
    J --> M
```

Data leakage — letting information from the test set influence training decisions, even indirectly — tends to make evaluation metrics look better than they would on genuinely new data. The safeguards above are designed to keep that risk low for every stage except the explicitly noted model-comparison step in Notebook 06.

## 19. Class Imbalance and Macro F1

| Class | Percentage |
|---|---|
| 0 (BENIGN) | 80.26% |
| 1 (Bot) | 0.17% |
| 2 (DDoS) | 11.44% |
| 3 (PortScan) | 8.12% |

With Class 0 making up over 80% of the data, a model could in principle achieve ~80% accuracy by predicting "BENIGN" for everything — accuracy alone would not reveal this. **Weighted metrics** are also dominated by class size, since they average per-class scores in proportion to support. **Macro F1** instead gives every class equal weight regardless of how many examples it has, so Class 1's poor F1 (0.790875) pulls the Macro F1 down noticeably (to 0.947471) even though it affects only 391 out of 223,737 test samples. This is precisely why Macro F1, not accuracy, was used as the primary optimization and comparison metric throughout the project — though it is not the *only* useful metric; class-wise precision/recall and the confusion matrix remain necessary for a complete picture.

## 20. End-to-End Machine Learning Pipeline

```mermaid
flowchart TD
    A[CIC-IDS2017] --> B[EDA]
    B --> C[Data Quality Analysis]
    C --> D[Stratified 80:20 Split]
    D --> E[Training-Only IQR Bounds]
    E --> F[IQR Capping]
    F --> G[Training-Based Correlation Filtering]
    G --> H[StandardScaler Fit on Training Data]
    H --> I[37 Preprocessed Features]
    I --> J[Variance Threshold]
    J --> K[Pearson Correlation Recheck]
    K --> L[Random Forest Feature Importance]
    L --> M[12 Selected Features]
    M --> N[Random Forest Baseline]
    N --> O[RandomizedSearchCV Tuning]
    O --> P[Model Comparison - 4 Algorithms]
    P --> Q[Decision Tree Selected]
    Q --> R[Final Evaluation]
    R --> S[Classification Report]
    R --> T[Confusion Matrix]
    R --> U[Error Analysis]
    R --> V[Feature Importance]
    S --> W[Saved Final Model]
    T --> W
    U --> W
    V --> W
    W --> X[Final Validation: True]
```

## 21. Final Project Results at a Glance

| Component | Result |
|---|---|
| Dataset | CIC-IDS2017 |
| Raw feature count | 78 numerical + 1 target |
| Feature count after cleaning | 68 |
| Features after correlation filtering (preprocessing) | 37 |
| Final selected features | 12 |
| Feature selection reduction (37→12) | 67.57% |
| Training samples | 894,944 |
| Evaluation samples | 223,737 |
| Final model | Decision Tree (`class_weight="balanced"`) |
| Training time | 6.36 seconds |
| Prediction time | 0.07 seconds |
| Accuracy | 99.9169% |
| Weighted Precision | 99.9175% |
| Weighted Recall | 99.9169% |
| Weighted F1 | 99.9172% |
| Macro F1 | 94.7471% |
| Correct predictions | 223,551 |
| Incorrect predictions | 186 |
| Overall error rate | 0.0831% |
| Most challenging class | Class 1 (Bot) |
| Class 1 F1-score | 79.0875% |
| Class 1 error rate | 20.2046% |
| Most important feature | Fwd Packet Length Mean |
| Top feature importance | 39.4534% |

## 22. Project Output Files

```
Network-Intrusion-Detection/
├── data/
│   ├── CIC_IDS2017.csv
│   └── processed/
│       ├── X_train.csv
│       ├── X_test.csv
│       ├── y_train.csv
│       ├── y_test.csv
│       ├── standard_scaler.pkl
│       ├── selected_features.csv
│       ├── X_train_selected.csv
│       ├── X_test_selected.csv
│       ├── model_building_baseline_results.csv
│       ├── random_forest_best_parameters.csv
│       ├── final_model_evaluation_results.csv
│       ├── final_class_wise_results.csv
│       ├── final_confusion_matrix.csv
│       └── final_feature_importance.csv
├── models/
│   ├── random_forest_baseline.pkl
│   ├── random_forest_tuned.pkl
│   └── final_decision_tree.pkl
└── notebooks/
    ├── 01_EDA.ipynb
    ├── 02_Preprocessing.ipynb
    ├── 03_Feature_Selection.ipynb
    ├── 04_Model_Building.ipynb
    ├── 05_Hyperparameter_Tuning.ipynb
    ├── 06_Model_Comparison.ipynb
    └── 07_Final_Model_Evaluation.ipynb
```

| File | Purpose |
|---|---|
| `standard_scaler.pkl` | Fitted `StandardScaler`, trained on training data only |
| `selected_features.csv` | Final 12 feature names with Random Forest importance scores |
| `random_forest_baseline.pkl` | Baseline Random Forest model (Notebook 04) |
| `random_forest_tuned.pkl` | Hyperparameter-tuned Random Forest (Notebook 05) |
| `random_forest_best_parameters.csv` | Best hyperparameters and CV score from `RandomizedSearchCV` |
| `final_decision_tree.pkl` | Final trained Decision Tree model (Notebook 07) |
| `final_model_evaluation_results.csv` | Final overall metrics (accuracy, precision, recall, F1, Macro F1, timings) |
| `final_class_wise_results.csv` | Final per-class precision/recall/F1/support |
| `final_confusion_matrix.csv` | Final 4×4 confusion matrix |
| `final_feature_importance.csv` | Final Decision Tree feature importances |

*Note: the notebooks themselves refer forward to a planned "Notebook 08 — Final Results & Visualization" for consolidated presentation. That notebook was not among the files provided for this documentation, so no content from it is represented here.*

## 23. Complete Project Mind Map

```mermaid
mindmap
  root((ML Network Intrusion Detection))
    Dataset
      CIC-IDS2017
      1,233,163 rows x 79 columns
    EDA
      Dataset Inspection
      Class Distribution
      Missing and Duplicate Detection
      Correlation Heatmap
    Preprocessing
      Cleaning: dedup, impute, constant removal
      Stratified 80:20 Split
      IQR Capping - train-derived bounds
      Correlation Filtering 0.90
      StandardScaler - train-only fit
    Feature Selection
      Variance Threshold
      Pearson Correlation Recheck
      Random Forest Importance - median cutoff
      37 to 12 features
    Modeling
      Random Forest Baseline
      RandomizedSearchCV Tuning
      Logistic Regression
      Decision Tree
      Random Forest
      Gradient Boosting
    Final Model
      Decision Tree
    Evaluation
      Accuracy 99.9169 percent
      Weighted F1 99.9172 percent
      Macro F1 94.7471 percent
      Classification Report
      Confusion Matrix
      Error Analysis
      Feature Importance
    Outputs
      Processed Datasets
      Trained Models
      Evaluation CSV Files
```

## 24. Technical Contributions

- Comprehensive EDA covering structure, quality, distribution, and correlation.
- Leakage-aware preprocessing: every fitted transformation (IQR bounds, correlation filter, scaler) learned from training data only.
- Stratified train-test splitting that preserved class proportions to two decimal places.
- IQR-based outlier **capping** rather than deletion, to protect minority-class samples.
- Correlation-based dimensionality reduction (68 → 37 features).
- Random Forest-based feature selection using a data-driven median-importance threshold (37 → 12 features), an overall 67.57% reduction.
- Class-imbalance-aware model configuration (`class_weight="balanced"` / `"balanced_subsample"`) across every model family tested.
- Macro F1 used consistently as the primary metric for tuning, comparison, and final evaluation.
- Randomized hyperparameter search with stratified cross-validation, including transparent reporting of a case where tuning did **not** improve held-out performance.
- Systematic comparison of four distinct model families rather than defaulting to one algorithm.
- Detailed class-wise performance analysis, confusion matrix analysis, and error-rate breakdown by class.
- Model-specific feature importance analysis, explicitly framed as non-causal.
- Model serialization and reproducible evaluation artifacts (CSV outputs for every result table).
- A final, explicit notebook-level validation step confirming all required files and metrics were correctly produced.

## 25. Limitations

- The dataset is severely imbalanced (Class 1 / Bot = 0.17% of all samples), and this imbalance is the dominant source of remaining classification error.
- Class 1 remains substantially harder to classify than the other three classes (F1 = 0.790875 vs. >0.999 for the others).
- Overall accuracy is strongly influenced by the majority class and can look excellent even when a minority class performs comparatively poorly — Macro F1 and class-wise metrics are necessary to see this.
- **Notebook 06 used the held-out evaluation dataset to compare and select among the four candidate models.** The final evaluation in Notebook 07 should therefore be described as performance on the dataset's held-out evaluation split, not as a completely untouched, unbiased test estimate.
- Feature importance values reflect the trained Decision Tree's reliance on each feature; they do not establish that any feature causes a particular type of intrusion.
- The hyperparameter search covered only 6 randomly sampled configurations from a defined search space, not an exhaustive search; the tuned model's held-out Macro F1 (0.946215) did not exceed the untuned baseline (0.946513), illustrating that CV improvement does not always transfer directly to test performance.
- All results are specific to CIC-IDS2017, this exact preprocessing pipeline, and this exact feature set — they do not automatically generalize to other intrusion-detection datasets or real-world network environments.
- No external validation on unseen network environments or newer traffic was performed as part of this project.

## 26. Future Scope

The following are possible directions for **future work**; none of them have been implemented in this project:

- Testing the pipeline on additional intrusion-detection datasets (e.g. CIC-IDS2018, UNSW-NB15).
- Validation on newer or live network traffic to assess concept drift.
- External validation on network environments not represented in CIC-IDS2017.
- Investigating more advanced imbalance-handling techniques (e.g. SMOTE-family oversampling, cost-sensitive ensembles) specifically targeted at the Bot class.
- Exploring stacked or blended ensemble models across the four algorithms already compared.
- Exploring deep learning approaches (e.g. sequence models on raw packet/flow sequences).
- Using SHAP or other explainability techniques for a more rigorous, model-agnostic view of feature contributions.
- Real-time or streaming intrusion detection deployment.
- API-based model serving for integration with network monitoring tools.
- Formal concept-drift monitoring for production use.
- Adversarial robustness testing against traffic specifically crafted to evade the model.
- Online/incremental learning to adapt the model as traffic patterns change over time.

## 27. Final Conclusion

This project developed an end-to-end supervised Machine Learning pipeline for network intrusion detection using the CIC-IDS2017 dataset. Exploratory data analysis was followed by leakage-aware preprocessing — stratified splitting, training-derived IQR capping, correlation-based filtering, and training-only standardization — and then by a three-stage feature-selection process that reduced the working feature space from 37 to 12 features (a 67.57% reduction) using Random Forest importance.

A Random Forest baseline was established and subsequently optimized using `RandomizedSearchCV` with stratified cross-validation; the tuned model's cross-validated Macro F1 (0.950453) did not fully translate to the held-out test set (0.946215), motivating a broader comparison across four model families. Among Logistic Regression, Decision Tree, Random Forest, and Gradient Boosting, the **Decision Tree** was selected within this project's evaluation criteria, based on the highest Macro F1 (0.947471) combined with markedly lower computational cost than its closest competitor, Gradient Boosting.

The final Decision Tree achieved **99.9169% accuracy** and **99.9172% weighted F1** on the held-out evaluation dataset, correctly classifying 223,551 of 223,737 samples (186 errors). Its **Macro F1 of 94.7471%** reflects strong performance on three of the four classes and a notably weaker result on the rare Bot class (Class 1), which remained the project's primary classification challenge throughout. The most important feature to the final model was **Fwd Packet Length Mean** (importance 0.394534), followed by Destination Port and Total Length of Bwd Packets.

Taken together, the project demonstrates a complete, reproducible Machine Learning workflow while being explicit about where its results are strong (majority and mid-sized classes), where they are weaker (the minority Bot class), and where the evaluation methodology itself has a documented caveat (test-set use during model comparison in Notebook 06). It is not presented as a perfect, production-ready, or universally superior solution.

## 28. Viva Quick Reference

**What dataset did you use?**
CIC-IDS2017, a network intrusion detection dataset with four traffic classes (BENIGN, Bot, DDoS, PortScan).

**What is the objective of the project?**
Build an end-to-end ML pipeline that classifies network traffic into four categories, with particular attention to class imbalance and leakage-free preprocessing.

**What type of ML problem is this?**
Supervised multi-class classification.

**Why is EDA necessary?**
To understand dataset structure, quality issues (missing/duplicate/constant values), class imbalance, and feature redundancy before deciding on preprocessing and modeling strategy.

**Why did you use an 80:20 split?**
It provides a large training set for a high-dimensional problem while keeping a statistically meaningful held-out evaluation set.

**Why did you use stratification?**
To preserve the (severely imbalanced) class proportions in both the training and testing subsets.

**What is class imbalance, and what are the actual class percentages?**
Some classes have far fewer samples than others. Here: Class 0 (BENIGN) 80.26%, Class 1 (Bot) 0.17%, Class 2 (DDoS) 11.44%, Class 3 (PortScan) 8.12%.

**Which class is most underrepresented?**
Class 1 (Bot), at 0.17% of the data.

**Why did you use IQR for outliers?**
It is a standard, distribution-agnostic way to flag statistically extreme values without assuming normality.

**Why cap outliers instead of deleting rows?**
To preserve dataset size and, critically, to avoid discarding already-scarce minority-class (Bot) samples.

**Why must IQR bounds be calculated using training data only?**
To prevent information about the test set's distribution from leaking into a preprocessing decision.

**Why did you use Pearson correlation, and why a 0.90 threshold?**
To detect linear redundancy between features; 0.90 is a common, fairly strict cutoff that flags only very strongly related pairs.

**What happened after correlation filtering in preprocessing vs. feature selection?**
Preprocessing removed 31 features (77 pairs above 0.90, 68→37). The recheck during feature selection on the smaller 23-feature set found 0 additional pairs above 0.90, so no further features were removed at that stage.

**Why did you use StandardScaler, and why fit only on training data?**
To bring features to a comparable scale; fitting only on training data avoids leaking test-set statistics into the transformation.

**What is data leakage, and how did you prevent it?**
Leakage is when information from outside the training set (especially the test set) influences model training or preprocessing decisions, inflating apparent performance. Prevented by fitting IQR bounds, correlation filters, and the scaler only on training data, and by using stratified CV (not the test set) for hyperparameter tuning.

**Why did you perform feature selection, and what was the full reduction path?**
To reduce redundancy and computational cost while retaining predictive signal: 37 → 23 (variance) → 23 (correlation recheck, no change) → 12 (Random Forest importance, median threshold).

**Which feature-selection methods were used, and why Random Forest for importance?**
Variance threshold, Pearson correlation, and Random Forest feature importance. Random Forest was used because it naturally produces an importance score per feature while handling non-linear relationships and interactions.

**What was the baseline model and its accuracy/Macro F1?**
Random Forest (`n_estimators=100, class_weight="balanced_subsample"`); accuracy 99.9057%, Macro F1 94.6513%.

**Why did you use Macro F1 as the primary metric?**
It weights all four classes equally regardless of size, so it reveals weak minority-class performance that accuracy or weighted metrics would hide.

**Which four models were compared, and what were their Macro F1 scores?**
Decision Tree (0.947471), Gradient Boosting (0.947336), Random Forest (0.946513), Logistic Regression (0.760096).

**Why was Decision Tree selected?**
Highest Macro F1 among the four, plus far lower training time (7.16s) than Gradient Boosting (558.28s), its closest competitor.

**What was the final model, and how many samples were used for training/evaluation?**
Decision Tree (`class_weight="balanced"`); trained on 894,944 samples, evaluated on 223,737.

**What was the final accuracy / Weighted F1 / Macro F1?**
Accuracy 99.9169%, Weighted F1 99.9172%, Macro F1 94.7471%.

**How many predictions were incorrect, and which class was hardest?**
186 of 223,737. Class 1 (Bot) was hardest, F1 = 0.790875, error rate ≈20.2%.

**What is a confusion matrix, and what does it tell you here?**
A table of actual vs. predicted classes; it shows that the main confusion is between Class 0 and Class 1, while Classes 2 and 3 are classified almost perfectly.

**Which feature had the highest importance, and what was its value?**
Fwd Packet Length Mean, importance 0.394534.

**What is the difference between weighted F1 and Macro F1?**
Weighted F1 averages per-class F1 weighted by class size (so it's dominated by the majority class); Macro F1 averages per-class F1 with equal weight regardless of size.

**What is the most important methodological caveat in the project?**
Notebook 06 used the held-out evaluation set to compare and select among the four candidate models, so the final evaluation should be described as performance on the held-out evaluation split rather than a completely untouched test estimate.

**What are the limitations, and what is the future scope?**
See Sections 25 and 26 above.

## 29. Final Project Logic (Plain-Language Walkthrough)

```
Raw CIC-IDS2017 data
  -> Understand the data using EDA
  -> Clean it (duplicates, missing/infinite values, constant columns) and encode the target
  -> Split into training and held-out evaluation data (stratified)
  -> Handle extreme values using training-derived IQR bounds
  -> Remove redundant, highly correlated information
  -> Standardize features without leaking test-set information
  -> Select the most informative 12 features
  -> Build a Random Forest baseline
  -> Tune Random Forest hyperparameters via cross-validation
  -> Compare four different algorithms on the same held-out data
  -> Select Decision Tree per this project's Macro F1 / efficiency criteria
  -> Evaluate the final model in detail
  -> Study class-wise performance (especially the rare Bot class)
  -> Analyze the 186 remaining errors
  -> Analyze feature importance (model-specific, non-causal)
  -> Save the final model and every evaluation artifact
  -> Run a final notebook-level validation check
```

## 30. Final Validation Status

Notebook 07 concluded with an explicit validation block, which produced:

| Check | Result |
|---|---|
| Required output files (model, overall/class-wise/confusion-matrix/feature-importance results) | FOUND |
| Metric validation (accuracy, precision, recall, F1, Macro F1 match saved results) | True |
| Prediction validation (223,737 predictions, 223,551 correct, confusion matrix shape 4×4) | True |
| Feature importance validation (12 features, importances sum to 1.000000) | True |
| **Final Notebook Validation** | **True** |

The notebook explicitly printed:

> `FINAL NOTEBOOK VALIDATION: True`
> `Notebook 07 Final Model Evaluation completed successfully.`

These checks confirm that the saved model file and every saved results file are internally consistent with the metrics reported in this document, and that no required artifact was missing at the end of the pipeline.

## 31. Final Master Project Flow

```mermaid
flowchart TD
    A[CIC-IDS2017] --> B[01 - EDA]
    B --> C[02 - Preprocessing]
    C --> D[03 - Feature Selection]
    D --> E[37 to 12 Features]
    E --> F[04 - Baseline Random Forest]
    F --> G[05 - Hyperparameter Tuning]
    G --> H[06 - Model Comparison]
    H --> I[Decision Tree Selected]
    I --> J[07 - Final Evaluation]
    J --> K[99.9169 percent Accuracy]
    K --> L[99.9172 percent Weighted F1]
    L --> M[94.7471 percent Macro F1]
    M --> N[Classification Report]
    N --> O[Confusion Matrix]
    O --> P[Error Analysis]
    P --> Q[Feature Importance]
    Q --> R[Final Model Saved]
    R --> S[Final Results Saved]
    S --> T[Validation: TRUE]
```

## 32. Final One-Line Summary

This project presents an end-to-end, leakage-aware Machine Learning pipeline for CIC-IDS2017 network intrusion detection, combining systematic preprocessing, feature selection, model comparison, and detailed class-wise evaluation to obtain a Decision Tree model with 99.9169% accuracy and 94.7471% Macro F1 on the held-out evaluation dataset — while transparently identifying the rare Bot class as its main remaining weakness.